## Задача. Разбиение на страты

Есть набор признаков. Используя эти признаки, нужно разбить объекты на страты так, чтобы дисперсия стратифицированного среднего была минимальна и доля каждой страты была не менее 5% от всех данных.

Разбиение должно быть простым, без использования ML. Можно использовать различные логические операции типа `>, <, >=, <=, ==, |, &` и преобразования `+, -, *, /`. Например, условие для одной из страт может быть таким `(x1 == 10) & (x3 + x5 < x6)`.

Данные разбиты на две части случайным образом. Первая часть будет предоставлена для исследования. Решение будет проверяться на второй части данных. Значения в столбцах `x1, ..., x10` - признаки, которые можно использовать для вычисления страт. Значения в столбце `y` - измерения, по которым будет вычисляться целевая метрика эксперимента.

Для получения 10 баллов достаточно получить дисперсию меньше 47000. Значения дисперсии больше 47000 штрафуются с шагом 400. Если минимальная доля страты меньше 0.05, то накладывается штраф с шагом 0.001. Подробнее функцию вычисления оценки смотрите в ячейке `#тесты`

In [138]:
import pandas as pd
import numpy as np


def get_strat(features: pd.Series) -> int:
    if features["x2"] <= 34:
        if features["x2"] <= 26:
            if features["x7"] <= 1999:
                if features["x10"] < 2:
                    return 0
                else:
                    return 1
            else:
                return 2
        else:
            if features["x10"] < 2:
                if features["x2"] <= 32:
                    if features["x8"] == 0:
                        return 3
                    else:
                        if features["x7"] <= 1993:
                            return 4
                        else:
                            return 5
                else:
                    return 6
            else:
                if features["x2"] <= 32:
                    if features["x8"] == 0:
                        if features["x3"] <= 30:
                            return 7
                        else:
                            return 8
                    else:
                        if features["x2"] <= 28:
                            return 9
                        else:
                            return 10
                else:
                    return 11
    else:
        if features["x2"] <= 37:
            if features["x6"] <= 2.65:
                return 12
            else:
                return 13
        else:
            return 14



def get_strats(df_features: pd.DataFrame) -> list:
    """
    Builds a bit-string (e.g. '1111') for each row based on the checks 
    in the original get_strat, then maps that bit-string to the final 
    numeric strat. Each bit corresponds to a condition in the order 
    they appear.
    """
    return df_features.apply(get_strat, axis=1)

In [139]:
# тесты
df = pd.read_csv("04-strat-hw-data-public.csv")
features = df.drop("y", axis=1)
df["strat"] = get_strats(features)


def calc_strat_params(df):
    """Вычисляет стратифицированную дисперсию и минимальную долю страт."""
    strat_vars = df.groupby("strat")["y"].var()
    weights = df["strat"].value_counts(normalize=True)
    stratified_var = (strat_vars * weights).sum()
    min_part = df["strat"].value_counts(normalize=True).min()
    return stratified_var, min_part


stratified_var, min_part = calc_strat_params(df)
print(f"var={stratified_var:0.0f}, min_part={min_part * 100:0.2f}%")

score_ = int(np.clip(10 - np.ceil((stratified_var - 47000) / 400), 0, 10))
penalty = int(np.clip(np.ceil((0.05 - min_part) * 1000), 0, 10))
score = max(score_ - penalty, 0)
print(f"score = max({score_}-{penalty}, 0) = {score}")

var=46746, min_part=5.04%
score = max(10-0, 0) = 10
